# Minesweeper

<pre>
    
</pre>

In [ ]:
import random

class Tile: 
    def __init__(self, row, col):
        self._row = row
        self._col = col

        self._ismine = False
        self._isrevealed = False
        self._hint = 0
        self._isflagged = False

    def get_row(self):
        return self._row
    def set_row(self, row):
        self._row = row

    def get_col(self):
        return self._col
    def set_col(self, col):
        self._col = col

    def get_ismine(self):
        return self._ismine
    def set_ismine(self, ismine):
        self._ismine = ismine

    def get_isrevealed(self):
        return self._isrevealed
    def set_isrevealed(self, isrevealed):
        self._isrevealed = isrevealed

    def get_isflagged(self):
        return self._isflagged
    def set_isflagged(self, isflag):
        self._isflagged = isflag

    def get_hint(self):
        return self._hint
    def set_hint(self, hint):
        self._hint = hint


    
class Game:
    def __init__(self, board_size:int, difficulty:int):
        self._board_size = board_size
        self._difficulty = [num for num in range(difficulty)]
        self._board = [[None for p in range(self._board_size)] for q in range(self._board_size)]
        self._win = False
        self._lose = False
        self._tiles_left = 0 # total num of tiles not mine left
        self._mines_left = 0 # total num of mines - number of flags
        self._directions = {
            'up': (-1,0),
            'down': (1,0),
            'left': (0,-1),
            'right': (0,1),
            'upleft': (-1,-1),
            'upright': (-1,1),
            'downleft': (1,-1),
            'downright': (1,1),
        }
        self._symbols = {
            'urtile' : '[ ]',
            'urmine' : '[X]',
            'flag' : '[F]',
        }
    
    def get_tile(self, coord:tuple): # Done return Tile object
        (row,col) = coord
        return self._board[row][col]

    def coord_isvalid(self, coord:tuple): # Done 
        (row,col) = coord
        # print(type(row),type(col))
        if 0 <= row < self._board_size and 0 <= col < self._board_size:
            return True
        
    def count_surr_mines(self, coord:tuple): # Done
        (row,col) = coord
        count = 0
        for direction in self._directions.values():
            y, x = direction
            new_row = row + y
            new_col = col + x
            new_coord = (new_row, new_col)
            if self.coord_isvalid(new_coord):
                neighbour = self.get_tile(new_coord)
                if neighbour.get_ismine():
                    count += 1
        return count

    def create_mines(self, coord:tuple): # Done
        # mine creation
        for row in range(self._board_size):
            for col in range(self._board_size):
                tile = Tile(row,col) 
                self._board[row][col] = tile

                if (row,col) != coord and random.randint(0,15) in self._difficulty: # tile is not tile user clicked, roll to set mine
                    (self._board[row][col]).set_ismine(True)
                    self._mines_left += 1
                else: 
                    (self._board[row][col]).set_ismine(False) # tile is set as not a mine
                    self._tiles_left += 1
        print("Mines have been set")
        print(f'{self._tiles_left} Tile objects created')

        # hint creation
        for row in range(self._board_size):
            for col in range(self._board_size): 
                # tile we are looking at is board[row][col]    
                self.get_tile((row,col)).set_hint(self.count_surr_mines((row, col)))
        print("Hints have been set")
        return True

    def display_board(self): # Done 
        print(f"Mines left: {self._mines_left}")
        line = f'     +{'-----+'*self._board_size}'
        output = '       '

        # create index row + first breakline
        for i in range(self._board_size): 
            output += f"{i}     "
        output+= '\n'+line+'\n'
        
        for row in range(self._board_size): 
            output += f" {row:2}  | "       # index column
            
            for col in range(self._board_size):
                tile = self.get_tile((row,col))

                if tile.get_isflagged() and not self._lose: # flags are displayed regardless of reveal status
                    output += f'{self._symbols['flag']} | '
                elif tile.get_isrevealed(): # for revealed tiles, display the hint 
                    if tile.get_hint() == 0:
                        output += '    | '
                    else: 
                        output += f' {tile.get_hint()}  | '
                else:                             # for unrevealed tile mines are X and nonmines are ' '
                    if self._lose and tile.get_ismine():
                        output += f'{self._symbols['urmine']} | '
                    else: 
                        output += f'{self._symbols['urtile']} | '

            output += '\n'+ line + '\n'
        print(output)
        return True
                    
    def get_user_coord(self): # Done returns tuple(row,col)
        try_count = 0
        while try_count < 10:
            print('Enter the coordinate in the form x,y: ')
            coord = input()
            print(f"Your input: {coord}")
            try:
                parts = coord.split(',')
                if len(parts) != 2:
                    raise ValueError
                col = int(parts[0])
                row = int(parts[1])
                if self.coord_isvalid((row,col)):
                    return (row, col)
                raise ValueError
            except ValueError:
                print("Please enter a valid coordinate.")
                try_count += 1
        print("Restart game.")
        return False

    def get_user_option(self): # Done returns "Dig" / "Flag" / "Unflag" / "Quit" 
        try_count = 0
        option_dict = {
            '1' : 'Flag',
            '2' : 'Unflag',
            '3' : 'Dig',
            '4' : 'Chord',
            '5' : 'Quit'
            }
        while try_count < 10:
            print('''
            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                ''')
            try:
                user_option = input()
                if user_option not in ('1','2','3','4','5'):
                    raise ValueError
                user_option = option_dict[user_option]
                print(f"Your option: {user_option}")
                return user_option
            except ValueError:
                print("Please enter a valid option number. ")
                try_count += 1
        print("Restart game.")
        return False

    def reveal_tile(self, coord:tuple): # Done
        tile = self.get_tile(coord)
        tile.set_isrevealed(True)
        return True

    def dig(self,coord:tuple): # Done 
        # checks and terminating case
        if not self.coord_isvalid(coord):
            return False
        tile = self.get_tile(coord)
        if tile.get_isrevealed():
            return False
        elif tile.get_isflagged(): # cannot dig on flag
            print("Unable to dig on flag")
            return False
        
        tile_ismine = tile.get_ismine() 
        if tile_ismine: # if tile is mine: game over
            print("You have hit a mine!")
            self._lose = True  
        else:           # if tile is not mine: 
            tile.set_isrevealed(True)
            self._tiles_left -= 1 
            hint = tile.get_hint()

            if hint==0: # dig all surrounding mines
                for direction in self._directions.values():
                    (row,col) = coord
                    y, x = direction
                    new_row = row + y
                    new_col = col + x
                    self.dig((new_row,new_col))
        return True

    def place_flag(self,coord:tuple): # Done
        tile = self.get_tile(coord)
        if tile.get_isflagged():
            print("Tile has already been flagged")
            return False
        elif tile.get_isrevealed():
            print("Tile cannot be flagged")
            return False
        else:
            tile.set_isflagged(True)
            self._mines_left -= 1
        return True

    def drop_flag(self,coord:tuple): # Done
        tile = self.get_tile(coord)
        if tile.get_isflagged():
            tile.set_isflagged(False)
        else: 
            print(f"There is no flag on tile {coord}")
            return False
        return True

    def chord(self, coord: tuple): # Done
        if not self.coord_isvalid(coord):
            return False
        tile = self.get_tile(coord)
        if not tile.get_isrevealed():# Only revealed hint tiles can be chorded
            return False
        hint = tile.get_hint()
        if hint == 0: # A zero tile has no need to be chorded
            return False
        row, col = coord
        flagged_count = 0
        for direction in self._directions.values(): # Count the flags surrounding the tile
            dy, dx = direction
            new_row = row + dy
            new_col = col + dx
            new_coord = (new_row, new_col)
            if self.coord_isvalid(new_coord):
                neighbour = self.get_tile(new_coord)
                if neighbour.get_isflagged():
                    flagged_count += 1

        # Only chord when the number of flags matches the hint
        if flagged_count != hint:
            print('Unable to chord tile. Insufficient flags')
            return False

        # Dig all adjacent unrevealed, unflagged tiles
        for direction in self._directions.values():
            dy, dx = direction
            new_row = row + dy
            new_col = col + dx
            new_coord = (new_row, new_col)

            if self.coord_isvalid(new_coord):
                neighbour = self.get_tile(new_coord)
                if not neighbour.get_isrevealed() and not neighbour.get_isflagged():
                    self.dig(new_coord)

        return True
    
    def run_game(self): 
        while not self._lose and not self._win and self._tiles_left > 0: 
            self.display_board () 
            coord = self.get_user_coord() # Done coord is tuple(row,col) 
            option = self.get_user_option() # Done option is "Dig" / "Flag" / "Quit" 
            if option == 'Dig':
                self.dig(coord) # Done
            elif option == 'Chord':
                self.chord(coord)
            elif option == 'Flag':
                self.place_flag(coord) ##################
            elif option == "Unflag":
                self.drop_flag(coord)
            elif option == 'Quit':
                self._lose = True
        
        self.display_board()
        if self._lose: 
            print('''
    ============================================================================
    ============================================================================
    #####     ###     #     #  #######         ###     #     #  #######  ####
    #        #   #    ##   ##  #              #    #   #     #  #        #    #
    #        #     #  # # # #  #              #     #  #     #  #        #    #
    #  ####  #     #  #  #  #  #####          #     #  #     #  #####    # #
    #     #  #######  #     #  #              #     #   #   #   #        #  #
    #     #  #     #  #     #  #              #     #    # #    #        #   #
    ######   #     #  #     #  #######         #####      #     #######  #    #
    ============================================================================
    ============================================================================
            ''')  
        elif self._tiles_left == 0:
            print('''
    ============================================================================
    ============================================================================
    #     #   ###     #     #        #     #  #######  #     #
    #     #  #    #   #     #        #     #     #     ##    #
     #   #   #     #  #     #        #     #     #     # #   #
       #     #     #  #     #        #  #  #     #     #  #  #
       #     #     #  #     #        # # # #     #     #   # #
       #     #     #  #     #        ##   ##     #     #    ##
       #      #####    #####         #     #  #######  #     #
    ============================================================================
    ============================================================================
    ''')
        return True

    def start(self):
        print("Insert the cordinate of the tile in the form x,y")
        start_coord = tuple((input()).split(','))
        # print(start_coord)
        try:
            # user inputs (col,row) but coord must be (row, col)
            col = int(start_coord[0])
            row = int(start_coord[1])
            start_coord = (row, col)
            
            self.create_mines(start_coord)
            self.dig(start_coord)
            self.run_game()
        except ValueError: 
            print("Enter a valid coordinate. ")
        return
            
### test

def minesweeper():
    print(r"""
                 +===========================+
                 |       MINESWEEPER         |
                 +===========================+

                        Find the safe tiles.
                    Avoid the hidden mines!

                        HOW TO PLAY
                    -------------------
                    Enter coordinates: x,y

                    1) Flag a tile
                    2) Unflag a tile
                    3) Dig a tile
                    4) Chord a tile
                    5) Quit game

                 +===========================+
                 |      GOOD LUCK!           |
                 +===========================+
    """)
    print("Please enter board size: ")
    boardsize = int(input())
    print(f"Boardsize: {boardsize}")
    if boardsize>=100: 
        print("Are you sure?")
        return
    print("Difficulty controls how many mines are generated. \nSelect difficulty from 1 to 10: ")
    difficulty = int(input())
    print(f"Difficulty: {difficulty}")
    g=Game(boardsize,difficulty)
    g.start()



minesweeper()


                 +===========================+
                 |       MINESWEEPER         |
                 +===========================+

                        Find the safe tiles.
                    Avoid the hidden mines!

                        HOW TO PLAY
                    -------------------
                    Enter coordinates: x,y

                    1) Flag a tile
                    2) Unflag a tile
                    3) Dig a tile
                    4) Chord a tile
                    5) Quit game

                 +===========================+
                 |      GOOD LUCK!           |
                 +===========================+
    
Please enter board size: 
Boardsize: 10
Difficulty controls how many mines are generated. 
Select difficulty from 1 to 10: 
Difficulty: 3
Insert the cordinate of the tile in the form x,y
Mines have been set
84 Tile objects created
Hints have been set
Mines left: 16
       0     1     2     3     4     5     6     7     8   

# Example Game: 

<pre>
                 +===========================+
                 |       MINESWEEPER         |
                 +===========================+

                        Find the safe tiles.
                    Avoid the hidden mines!

                        HOW TO PLAY
                    -------------------
                    Enter coordinates: x,y

                    1) Flag a tile
                    2) Unflag a tile
                    3) Dig a tile
                    4) Chord a tile
                    5) Quit game

                 +===========================+
                 |      GOOD LUCK!           |
                 +===========================+
    
Please enter board size: 
Boardsize: 9
Difficulty controls how many mines are generated. 
Select difficulty from 1 to 10: 
Difficulty: 3
Insert the cordinate of the tile in the form x,y
Mines have been set
71 Tile objects created
Hints have been set
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 8,8

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Dig
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 6,7

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Flag
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 6,8

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Dig
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 6,8

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Chord
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] | [ ] |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 5,7

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Chord
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 6,5

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Dig
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 5,5

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Flag
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 6,5

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Chord
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 0,0

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Dig
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 1,1

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Flag
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  | [ ] | [F] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 1,0

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Dig
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  | [ ] | [F] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 0,1

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Dig
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  | [ ] | [ ] |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 0,1

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Chord
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 2,0

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Dig
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 2,0

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Chord
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  | [ ] | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  | [ ] | [ ] | [ ] | [ ] | [ ] | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  | [ ] | [ ] | [ ] | [ ] |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 0,2

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Chord
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 2,3

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Flag
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 1,1

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Unflag
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [ ] |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 1,1

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Flag
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 3,4

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Chord
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [ ] |  1  |     |  2  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  |  1  |  1  |     |  2  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  |  1  |     |     |     |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] |  1  |     |     |     |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 4,0

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Flag
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [F] |  1  |     |  2  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  |  1  |  1  |     |  2  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  |  1  |     |     |     |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] |  1  |     |     |     |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 8,0

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Flag
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [F] |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  |  1  |  1  |     |  2  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  |  1  |     |     |     |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] |  1  |     |     |     |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 8,1

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Flag
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [F] |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  |  1  |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  |  1  |     |     |     |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] |  1  |     |     |     |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 7,2

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Chord
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [F] |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  |  1  |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [ ] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 8,4

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Flag
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [F] |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  |  1  |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [ ] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 3,7

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Flag
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [F] |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  |  1  |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] | [ ] | [ ] | [F] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 2,6

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Chord
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [F] |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  |  1  |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [ ] |  1  |  1  | [F] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 0,7

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Flag
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [F] |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  |  1  |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [F] |  1  |  1  | [F] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  | [ ] | [ ] | [ ] | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 1,7

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Chord
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [F] |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  |  1  |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [F] |  1  |  1  | [F] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  |  1  |  1  |  1  | [ ] |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+

Enter the coordinate in the form x,y: 
Your input: 3,8

            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Chord
            5) Quit
                
Your option: Dig
       0     1     2     3     4     5     6     7     8     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  0  |  1  |  1  |  1  |  1  | [F] |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  1  |  1  | [F] |  1  |  1  |  1  |  1  |     |  2  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  2  |  1  |  2  |  2  |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  3  |     |  1  | [F] |  1  |     |     |     |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  4  |     |  1  |  1  |  1  |  1  |  1  |  1  |  1  | [F] | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  5  |     |     |     |     |  1  | [F] |  1  |  1  |  1  | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  6  |  1  |  1  |  1  |  1  |  2  |  2  |  2  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  7  | [F] |  1  |  1  | [F] |  1  |  1  | [F] |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+
  8  |  1  |  1  |  1  |  1  |  1  |  1  |  1  |  1  |     | 
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+


    ============================================================================
    ============================================================================
    #     #   ###     #     #        #     #  #######  #     #
    #     #  #    #   #     #        #     #     #     ##    #
     #   #   #     #  #     #        #     #     #     # #   #
       #     #     #  #     #        #  #  #     #     #  #  #
       #     #     #  #     #        # # # #     #     #   # #
       #     #     #  #     #        ##   ##     #     #    ##
       #      #####    #####         #     #  #######  #     #
    ============================================================================
    ============================================================================
    
</pre>

True
<class 'int'>
